In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans, DBSCAN
import matplotlib.pyplot as plt
import seaborn as sns
import pickle












### 4. Кластеризация

# Нормализация данных для кластеризации



In [ ]:
with open('shop_dataset.pkl', 'rb') as f:
    data = pickle.load(f)

data['Sales_Date'] = pd.to_datetime(data['Sales_Date'])
data['Month'] = data['Sales_Date'].dt.month
data['Weekday'] = data['Sales_Date'].dt.weekday
data['Holiday'] = data['Seasonality'].apply(lambda x: 1 if x in ['Black Friday', 'Summer Sales'] else 0)

In [ ]:
#Кодирование категориальных 
encoder = LabelEncoder()
data['Region'] = encoder.fit_transform(data['Region'])
data['Seasonality'] = encoder.fit_transform(data['Seasonality'])
X = data[['Region', 'Seasonality', 'Competitor_Price', 'Discount', 'Month', 'Weekday', 'Holiday']]
y = data['Demand_Forecast']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#Decision Treeദ്ദി(˵ •̀ ᴗ - ˵ ) ✧
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
mae_dt = mean_absolute_error(y_test, y_pred_dt)

In [ ]:
#Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

In [ ]:
#Gradient Boosting ૮ ˶ᵔ ᵕ ᵔ˶ ა
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
mae_gb = mean_absolute_error(y_test, y_pred_gb)

In [ ]:
print(f"Decision Tree MAE: {mae_dt}")
print(f"Random Forest MAE: {mae_rf}")
print(f"Gradient Boosting MAE: {mae_gb}")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X[['Region', 'Competitor_Price', 'Discount', 'Month']])

In [ ]:
#кластеризация K-Means -__-
kmeans = KMeans(n_clusters=3, random_state=42)
data['Cluster_KMeans'] = kmeans.fit_predict(X_scaled)


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=data, x='Competitor_Price', y='Demand_Forecast', hue='Cluster_KMeans', palette='viridis')
plt.title("K-Means Clustering of Regions Based on Demand Patterns")
plt.show()

In [ ]:
#кластеризация DBSCAN +__+
dbscan = DBSCAN(eps=0.5, min_samples=5)
data['Cluster_DBSCAN'] = dbscan.fit_predict(X_scaled)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=data, x='Competitor_Price', y='Demand_Forecast', hue='Cluster_DBSCAN', palette='coolwarm')
plt.title("DBSCAN Clustering of Regions Based on Demand Patterns")
plt.show()